# Tiền xử lý và chia dữ liệu — Elden Ring Steam Reviews

Notebook này tạo dữ liệu văn bản sạch ở mức cơ bản và chia train/validation/test có phân tầng. Chỉ dùng `review` làm đầu vào và `voted_up` làm nhãn; không dùng metadata khác, không cân bằng dữ liệu và không huấn luyện mô hình. Tệp CSV gốc không bị thay đổi.

In [1]:
from html import unescape
from pathlib import Path
import re

import pandas as pd
from sklearn.model_selection import train_test_split

DATA_PATH = Path('data/raw/elden_ring_steam_reviews.csv')
OUTPUT_DIR = Path('data/processed')
if not DATA_PATH.exists():
    DATA_PATH = Path('../data/raw/elden_ring_steam_reviews.csv')
    OUTPUT_DIR = Path('../data/processed')
if not DATA_PATH.exists():
    raise FileNotFoundError('Không tìm thấy data/raw/elden_ring_steam_reviews.csv')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
required_columns = {'review', 'voted_up'}
if not required_columns.issubset(df.columns):
    raise ValueError(f'Thiếu cột bắt buộc: {required_columns - set(df.columns)}')
print(f'Dữ liệu gốc: {df.shape[0]:,} dòng × {df.shape[1]} cột')

Dữ liệu gốc: 9,794 dòng × 16 cột


## 1. Loại đánh giá không có nội dung và tạo nhãn
Coi ô thiếu, chuỗi rỗng và chuỗi chỉ có khoảng trắng là đánh giá trống. Chuyển `voted_up` thành nhãn số: True → 1 (đề xuất), False → 0 (không đề xuất). Dừng với lỗi nếu gặp nhãn ngoài hai giá trị dự kiến để tránh âm thầm làm mất dữ liệu.

In [2]:
has_review = df['review'].notna() & df['review'].astype('string').str.strip().ne('').fillna(False)
removed_count = int((~has_review).sum())
data = df.loc[has_review, ['review', 'voted_up']].copy()

label_map = {'true': 1, 'false': 0}
labels = data['voted_up'].astype('string').str.strip().str.lower().map(label_map)
if labels.isna().any():
    unexpected = data.loc[labels.isna(), 'voted_up'].unique().tolist()
    raise ValueError(f'Giá trị voted_up không hợp lệ: {unexpected}')
data['label'] = labels.astype('int8')
print(f'Số đánh giá trống đã loại: {removed_count}')
print(f'Số mẫu còn lại: {len(data):,}')
print('Phân phối nhãn trước khi chia:')
display(data['label'].value_counts().sort_index().rename(index={0: '0 - Không đề xuất', 1: '1 - Đề xuất'}).to_frame('Số mẫu'))

Số đánh giá trống đã loại: 31
Số mẫu còn lại: 9,763
Phân phối nhãn trước khi chia:


,Số mẫu
label,
0 - Không đề xuất,591
1 - Đề xuất,9172


**Nhận xét:** Loại bỏ các đánh giá không có nội dung trước khi chia; nhãn lấy trực tiếp từ khuyến nghị Steam. Chỉ giữ hai cột cần thiết, tránh đưa thông tin tác giả, thời gian chơi hoặc các thuộc tính khác vào mô hình.

## 2. Làm sạch văn bản cơ bản
Giải mã HTML entity, bỏ thẻ HTML nếu có và chuẩn hóa chuỗi khoảng trắng. Không chuyển chữ thường, không bỏ dấu câu và không xóa stopword để giữ nguyên các phủ định như `not`, `no`, `never`; cách làm này cũng không gắn với một loại mô hình cụ thể.

In [3]:
def clean_review(text):
    """Làm sạch cấu trúc HTML/khoảng trắng, giữ nguyên nội dung và phủ định."""
    text = unescape(str(text))
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

data['review'] = data['review'].map(clean_review)
# Bảo đảm bước làm sạch không để lại văn bản rỗng.
if data['review'].str.strip().eq('').any():
    raise ValueError('Có đánh giá trở thành rỗng sau khi làm sạch; cần kiểm tra dữ liệu nguồn.')
print('Ví dụ sau khi làm sạch:')
display(data[['review', 'label']].head())
print('Kiểm tra phủ định:')
print(clean_review('This is not bad, no issues, never forget!'))

Ví dụ sau khi làm sạch:


,review,label
0,being killed over and over again is fun,1
1,I write this review as I have 100% completed E...,1
2,Fun,1
3,pretty cool.,1
4,AMAZINGGGGGGGGGGGGG,1


Kiểm tra phủ định:
This is not bad, no issues, never forget!


## 3. Chia train / validation / test
Chia 70% train và 30% phần tạm; tiếp tục chia phần tạm đều thành validation và test. Cả hai lần đều phân tầng theo nhãn và sử dụng `random_state=42`. Không cân bằng lớp trước khi chia.

In [4]:
X = data['review']
y = data['label']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_validation, X_test, y_validation, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

splits = {
    'train': pd.DataFrame({'review': X_train, 'label': y_train}),
    'validation': pd.DataFrame({'review': X_validation, 'label': y_validation}),
    'test': pd.DataFrame({'review': X_test, 'label': y_test}),
}

# Kiểm tra kích thước và tính toàn vẹn trước khi lưu.
assert sum(len(part) for part in splits.values()) == len(data)
assert set().union(*(set(part.index) for part in splits.values())) == set(data.index)
assert sum(len(part.index.intersection(other.index)) for name, part in splits.items()
           for other_name, other in splits.items() if name < other_name) == 0

summary = []
for name, part in splits.items():
    counts = part['label'].value_counts().reindex([0, 1], fill_value=0)
    summary.append({
        'Tập dữ liệu': name,
        'Số mẫu': len(part),
        'Không đề xuất (0)': int(counts[0]),
        'Đề xuất (1)': int(counts[1]),
        'Tỷ lệ đề xuất (%)': round(counts[1] / len(part) * 100, 2),
    })
display(pd.DataFrame(summary).set_index('Tập dữ liệu'))

,Số mẫu,Không đề xuất (0),Đề xuất (1),Tỷ lệ đề xuất (%)
Tập dữ liệu,,,,
train,6834,414,6420,93.94
validation,1464,88,1376,93.99
test,1465,89,1376,93.92


**Nhận xét:** Phân tầng giúp tỷ lệ nhãn trong mỗi tập gần với tỷ lệ toàn bộ dữ liệu; các tập tách biệt và bao phủ toàn bộ mẫu còn lại. Kích thước từng tập có thể lệch tối đa một vài mẫu do số lượng phải là số nguyên.

## 4. Lưu các tập dữ liệu
Lưu ba CSV chỉ gồm `review` và `label` vào `data/processed/`.

In [5]:
for name, part in splits.items():
    output_path = OUTPUT_DIR / f'{name}.csv'
    part.reset_index(drop=True).to_csv(output_path, index=False, encoding='utf-8')
    print(f'Đã lưu {output_path} ({len(part):,} dòng)')

print('Các tập đã được lưu. Dữ liệu gốc không bị thay đổi.')

Đã lưu ..\data\processed\train.csv (6,834 dòng)
Đã lưu ..\data\processed\validation.csv (1,464 dòng)
Đã lưu ..\data\processed\test.csv (1,465 dòng)
Các tập đã được lưu. Dữ liệu gốc không bị thay đổi.
